# Allons plus loin avec nos modèles
Maintenant que nous avons une vue d'ensemble des différentes étapes qui constituent la création de notre modèle, nous allons voir quelles que astuces, fonctionalités avancées et outils pouvant être utiles lorsque l'on entraîne notre modèle.

## Les époques et la fonction de perte (epoch & loss function)
Dans le notebook précédent, nous avons pu expliquer le fonctionnement des différentes fonctions de pertes et l'idée générale qui est de minimiser la perte au maximum pour avoir de meilleurs résultats.

Nous avons introduit le concept d'époques à la fin du notebook `Quickstart.ipynb`, cela correspond au nombre de fois que l'on passe l'ensemble des données d'entraînement dans le modèle pour l'entraîner.

Dans notre exemple, nous avions utilisé une valeur fixe et arbitraire. En pratique cette valeur se trouve expérimentalement, il faut tester et augmenter le nombre d'époques tant que la perte calculée sur les données de validation diminue.
Si cette dernière commence à augmenter alors que la celle obtenue sur les données d'entraînement continue à diminuer, on atteint un cas d'**overfitting** ou surapprentissage.

Dans ce cas, le modèle se met à perdre sa capacité à généraliser. Ainsi, bien qu'il aura d'encore meilleurs résultats sur les données d'entraînement, lorsque l'on le teste sur d'autres données telles que celles de vérification, il aura des résultats de moins en moins bons.
Or la généralisation est ce que l'on cherche à obtenir avec ces réseaux de neurones, il est donc nécessaire de trouver un nombre d'époques qui permet d'avoir les meilleurs performances possibles sans atteindre l'overfitting.


## Outils de visualisation

Afin de voir en temps réel l'évolution de la perte d'entraînement et de validation, on utilise souvent des outils comme **TensorBoard**.

In [1]:
#pip install tensorboard

In [5]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor
from torch.utils.tensorboard import SummaryWriter
from datetime import datetime

torch.set_num_threads(16)

# -----------------------------
# On définit le modèle
# -----------------------------
class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits


# -----------------------------
# Création du pipeline 
# -----------------------------
class ModelPipeline:
    def __init__(self):
        # Récupération des bases de données et transformation en Dataloaders
        self.training_data = datasets.FashionMNIST(
            root="data", train=True, download=True, transform=ToTensor()
        )
        self.test_data = datasets.FashionMNIST(
            root="data", train=False, download=True, transform=ToTensor()
        )
        self.batch_size = 64

        self.train_dataloader = DataLoader(self.training_data, batch_size=self.batch_size, shuffle=True)
        self.test_dataloader = DataLoader(self.test_data, batch_size=self.batch_size)

        # Détecte s'il est possible d'utiliser un GPU ou si les calculs se feront sur CPU
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"Utilisation de {self.device}")

        # Création du modèle, de la fonction de perte et l'optimiseur
        self.model = NeuralNetwork().to(self.device)
        self.loss_fn = nn.CrossEntropyLoss()
        self.optimizer = torch.optim.SGD(self.model.parameters(), lr=1e-3)

    def train(self):
        self.model.train()
        total_loss, correct = 0, 0

        for X, y in self.train_dataloader:
            X, y = X.to(self.device), y.to(self.device)

            pred = self.model(X)
            loss = self.loss_fn(pred, y)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        avg_loss = total_loss / len(self.train_dataloader)
        accuracy = correct / len(self.training_data)
        return avg_loss, accuracy

    def test(self):
        self.model.eval()
        total_loss, correct = 0, 0

        with torch.no_grad():
            for X, y in self.test_dataloader:
                X, y = X.to(self.device), y.to(self.device)
                pred = self.model(X)
                total_loss += self.loss_fn(pred, y).item()
                correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        avg_loss = total_loss / len(self.test_dataloader)
        accuracy = correct / len(self.test_data)
        return avg_loss, accuracy

    def run(self, epochs=20):
        # N'affiche qu'une toutes les 5 époques, afin de ne pas surcharger le notebook
        epoch_to_print = epochs // 5

        run_name = f"fashionMNIST_SGD_lr1e-3_{datetime.now().strftime('%Y-%m-%d_%H-%M-%S')}"
        writer = SummaryWriter(log_dir=f"runs/{run_name}")
        for epoch in range(epochs):
            train_loss, train_acc = self.train()
            val_loss, val_acc = self.test()

            writer.add_scalar("Perte d'entraînement", train_loss, epoch)
            writer.add_scalar("Perte de validation", val_loss, epoch)
            writer.add_scalar("Précision données d'entraînement", train_acc, epoch)
            writer.add_scalar("Précision données de validation", val_acc, epoch)

            if epoch % epoch_to_print == 0:
                print(f"Époque n°{epoch+1}/{epochs}")
                print(f"  Perte d'entraînement: {train_loss:.4f} | Précision sur les données d'entraînement: {train_acc:.3f}")
                print(f"  Perte de validation: {val_loss:.4f} | Précision sur les données de validation: {val_acc:.3f}\n")

        writer.close()
        print("Fin de l'entraînement")


In [3]:
%load_ext tensorboard
%tensorboard --logdir runs

In [7]:
test = ModelPipeline()
test.run(epochs=500)


Utilisation de cpu
Époque n°1/500
  Perte d'entraînement: 2.2459 | Précision sur les données d'entraînement: 0.318
  Perte de validation: 2.1724 | Précision sur les données de validation: 0.422

Époque n°101/500
  Perte d'entraînement: 0.3980 | Précision sur les données d'entraînement: 0.862
  Perte de validation: 0.4370 | Précision sur les données de validation: 0.844

Époque n°201/500
  Perte d'entraînement: 0.3340 | Précision sur les données d'entraînement: 0.884
  Perte de validation: 0.3863 | Précision sur les données de validation: 0.865

Époque n°301/500
  Perte d'entraînement: 0.2898 | Précision sur les données d'entraînement: 0.899
  Perte de validation: 0.3609 | Précision sur les données de validation: 0.871

Époque n°401/500
  Perte d'entraînement: 0.2523 | Précision sur les données d'entraînement: 0.912
  Perte de validation: 0.3408 | Précision sur les données de validation: 0.878

Fin de l'entraînement


## Explication du code

Comme vous avez pu le remarquer, cette classe `ModelPipeline`réutilise le code que nous avons utilisé dans le notebook Quickstart tout en le rendant plus modulaire et en intégrant TensorBoard.

Voici les étapes pour utiliser TensorBoard:
1. Installer TensorBoard avec pip: `pip install tensorboard`
2. Importer la classe `SummaryWriter`de cette façon: `from torch.utils.tensorboard import SummaryWriter`
3. À l'aide de cette classer, créer l'objet `writter`permettant de communiquer les noms et valeurs souhaités à TensorBoard. On peut spécifier le dossier dans lesquels les logs de l'exécution seront enregistrés: `writer = SummaryWriter(log_dir=f"runs/{run_name}")`
4. Ajouter la valeur des métriques voulues avec `add_scalar("name_of_the_metrics", x_metric,y_metric)`
5. Fermer l'objet `writer`à la fin de l'entraînement. L'écriture des données est finie.

Avec jupyter-notebook, avant d'exécuter notre pipeline il faut lancer la commande `%load_ext tensorboard ` qui va lancer TensorBoard et la commande `tensorboard --logdir runs`qui va lui indiquer dans quel dossier se trouvent les métriques que nous souhaitons afficher.

## Comparer les différentes fonctions de pertes et les optimiseurs

Comme mentionné précédemment, il est possible de spécifier avec `--logdir` le dossier que nous voulons charger dans TensorBoard.
Ce sont les metrics de toutes les sessions qui seront chargées. Ainsi, il est possible de lancer le pipeline plusieurs fois en changer la configuration du modèle et comparer les performances de chacune d'entres elles.

Pour ce faire, il faut rendre la classe pipeline un peu plus modulable:

In [6]:
from torch import optim
from torchvision import transforms
import inspect
from datetime import datetime

class NewModelPipeline:
    def __init__(
        self,
        model: nn.Module,
        dataset_fn,
        optimizer: torch.optim.Optimizer,
        loss_fn: nn.Module,
        batch_size: int = 64,
        transform=None,
        log_root: str = "runs",
        log_name: str = ""
    ):
        # Configuration des bases de données et des Dataloaders
        transform = transform or transforms.ToTensor()
        self.train_data = dataset_fn(root="data", train=True, download=True, transform=transform)
        self.test_data = dataset_fn(root="data", train=False, download=True, transform=transform)
        self.train_loader = DataLoader(self.train_data, batch_size=batch_size, shuffle=True)
        self.test_loader = DataLoader(self.test_data, batch_size=batch_size)

        # Choix GPU ou CPU
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        print(f"J'utilise {self.device}")

        # Le modèle, l'optimiseur et la fonction de perte
        self.model = model.to(self.device)
        self.optimizer = optimizer
        self.loss_fn = loss_fn

        # Laisse la possibilité de nommer le log de cette run
        if log_name == "":
            # Auto nomme pour les logs de TensorBoard 
            dataset_name = dataset_fn.__name__
            optimizer_name = self.optimizer.__class__.__name__
            loss_name = self.loss_fn.__class__.__name__
    
            timestamp = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")
            log_name = f"{dataset_name}_{optimizer_name}_{loss_name}_{timestamp}"
            
        self.writer = SummaryWriter(log_dir=f"{log_root}/{log_name}")

    # Étape d'entraînement
    def train_one_epoch(self):
        self.model.train()
        total_loss, correct = 0, 0
        for X, y in self.train_loader:
            X, y = X.to(self.device), y.to(self.device)

            pred = self.model(X)
            loss = self.loss_fn(pred, y)

            self.optimizer.zero_grad()
            loss.backward()
            self.optimizer.step()

            total_loss += loss.item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        avg_loss = total_loss / len(self.train_loader)
        accuracy = correct / len(self.train_loader.dataset)
        return avg_loss, accuracy

    # Étape d'évaluation
    def evaluate(self):
        self.model.eval()
        total_loss, correct = 0, 0
        with torch.no_grad():
            for X, y in self.test_loader:
                X, y = X.to(self.device), y.to(self.device)
                pred = self.model(X)
                total_loss += self.loss_fn(pred, y).item()
                correct += (pred.argmax(1) == y).type(torch.float).sum().item()

        avg_loss = total_loss / len(self.test_loader)
        accuracy = correct / len(self.test_loader.dataset)
        return avg_loss, accuracy

    # Boucle d'entraînement complète
    def fit(self, epochs=10, interactive=False):
        epoch_to_print = epochs // 5
        for epoch in range(epochs):
            train_loss, train_acc = self.train_one_epoch()
            val_loss, val_acc = self.evaluate()

            if epoch % epoch_to_print == 0:
                print(f"Époque {epoch+1}/{epochs}")
                print(f"  Perte d'entraînement: {train_loss:.4f} | Précision d'entraînement: {train_acc:.3f}")
                print(f"  Perte de validation: {val_loss:.4f} | Précision de validation: {val_acc:.3f}\n")

  
            self.writer.add_scalar("Perte/entrainement", train_loss, epoch)
            self.writer.add_scalar("Perte/validation", val_loss, epoch)
            self.writer.add_scalar("Précision/entrainement", train_acc, epoch)
            self.writer.add_scalar("Précision/validation", val_acc, epoch)

            if interactive:
                cont = input("Continuer judqu'à la prochaine époque? (o/n): ")
                if cont.lower() != "o":
                    print("Entraînement arrêté.")
                    break

        self.writer.close()


Cette classe nous permet de maintenant pouvoir choisir quelle **base de données**, quel **optimiseur** et quelle **fonction de perte** nous souhaitons utiliser. Le nom donné au dossier contenant les logs de cette exécution sera automatiquement généré à l'aide de ces informations si aucun autre nom n'est donné.
Il est également possible de choisir quelle **classe de modèle** et la **taille d'échantillonnage** nous souhaitons utiliser.

J'ai également ajouté la possibilité d'utiliser un **mode interactif** permettant de contôler l'enchaînement des passes de façon plus précise.

Ci-dessous se trouve une instance de notre pipeline en utilisant l'optimiseur **Adam**.

In [ ]:
# Initialize the tensorboard 

%reload_ext tensorboard
%tensorboard --logdir runs


In [ ]:
# Définie d'abord les composants / paramètres que nous allons fournir au pipeline. Certains seront réutilisés par la suite.
normal_model = NeuralNetwork()
loss_x_entropy = nn.CrossEntropyLoss()
adam_optimizer = optim.Adam(normal_model.parameters(), lr=1e-3) 
# Création du pipeline pipeline
pipeline = NewModelPipeline(
    model=normal_model,
    dataset_fn=datasets.FashionMNIST,
    optimizer=adam_optimizer,
    loss_fn=loss_x_entropy,
    batch_size=64,
)


# Lance l'entraînement. Par défaut le mode interactif est désactivé
pipeline.fit(epochs=50)

Nous pouvons maintenant facilement comparer les deux optimiseurs avec **TensorBoard**.

Bien qu'il semble que dans le cas de la **SGD** avec un taux d'apprentissage de 0.001 les performances plafonnent vers une précision de 84% et même après 80 époques il ne semble pas y avoir de surapprentissage, en utilisant **Adam** avec le même taux d'apprentissage le modèle atteint une précision dépassant les **90%** mais surapprends après une dizaine d'époques.

![accuracy_training](../img/Accuracy_training_FashionMNIST_Adam.png)
![loss_validation](../img/Loss_validation_FashionMNIST_Adam.png)

## La classe du modèle de réseau de neurones

Jusqu'à présent nous avions expliqué à quoi correspondaient les couches **ReLU**, **Linear** (linéaires), et **Flatten** (l'applatissement d'une matrice 2D repésentant une image et une liste) mais sans vraiment expliquer pourquoi nous construisons ainsi le modèle.

Voici un schémas simplifié pour un réseau prenant en entrée une image de nuances de gris de taille $2*2$, ayant 2 couches/layers cachées de 5 neurones et 2 choix en sortie:
![schema of our neural network](../img/BasicNeuralNetwork.png)

Dans notre classe actuelle, la premier layer est un appliquant une transformation linéaire. La taille d'entrée est de $28*28*1=784$ neurones car les images en entrée sont des nuances de gris de taille 28x28 càd avec un seul canal.
Dans le cas d'une image RVB, la taille d'entrée aurait été de $28*28*3$.

La sortie de ce layer fait **512 neurones**, de même pour le 2nd layer. Ce nombre est choisi **arbitrairement**, c'est un des **hyperparamètres** que l'on peut modifier. Ce dernier définit la capacité que le réseau aura: plus elle est grande, plus d'informations pouront être représentées et interprêtées par ce réseau.
On choisit souvent une **puissance de 2** afin d'obtenir de meilleurs performances sur GPU.

Si la valeur est trop basse, le réseau n'arrivera pas à apprendre les patterns complexes des données, c'est du sous-apprentissage ou **underfitting**. Si au contraite la taille est trop importante, le réseau peut échouer à généraliser et ne sera performant que sur ses données d'entraînement, c'est du surapprentissage ou **overfitting**.

En pytorch, le layer linéaire est défini en lui fournissant une taille d'entrée et une taille de sortie.
Comme indiqué sur le schéma, ça correspond à la relation entre deux couches de neurones.

On explique la structure suivante définie plus haut:

```python
self.linear_relu_stack = nn.Sequential(
    nn.Linear(28*28, 512),
    nn.ReLU(),
    nn.Linear(512, 512),
    nn.ReLU(),
    nn.Linear(512, 10)
)
```

Le 1er layer en pytorch relie *la donnée fournie en entrée au réseau* avec la première couche cachée ou **hidden layer**. On trouve ensuite le layer ReLU qui correspond à l'application de la fonction `f()` dans le schéma.

Le 2e layer pytorch relie le *1er hidden layer** avec le *2e hidden layer*. On applique à nouveau le layer Relu. L'association du layer linéaire et du ReLU correspond à la couche de neurones annotés $ f(Wx + b)$ sur le schémas.

Enfin, le 3e layer pytorch relie le *2e hidden layer* avec les valeurs de sortie possibles.

Il faut donc être très prudent lorsque l'on parle de layer car en fonction que ce soit la fonction pytorch ou la repésentation schématique que l'on en fait, on ne parle pas exactement des même choses.

### Dois-je et puis-je changer le nombre de neurones dans mes hidden layers

Habituellement prendre une valeur de 512 neurones fais parfaitement l'affaire mais il y a des cas dans lesquels on souhaiterais changer cette valeur pour obtenir de meilleures performances. La règle est la suivante:
* Si lors l'on teste le modèle la perte d'entraînement et celle de validation sont trop élevés, le modèle peut **underfit**, il n'y a pas assez de neurones pour comprendre les patterns complexes. Ainsi, il faut **augmenter** la taille du hidden layer.
* Si la perte d'entraînement est basse MAIS celle de validation est haute, le modèle peut **overfit**, il a trop de neurones et n'arrive plus à généraliser. Dans ce cas, au contraire, il est préférable de **diminuer** la taille du hidden layer.

Il est également important de noté que même lorsque l'on n'est pas en cas de surapprentissage, si l'entrainement est lent et instable, le problème peut être lié au fait qu'il y ait trop de paramètre, rendant la convergence plus difficile. Dans ce cas il peut être utile de **diminuer** la taille du layer.

### Puis-je changer le nombre de layers?

Absolument! Il est même possible de n'utiliser que 2 couches linéaires, résultant à un unique hidden layer. Il est par contre déconseillé de faire cela.

Chaque couche permet au modèle de détecter plus de caractéristiques abstraites donc limiter ce nombre risque de grandement restreindre les patterns reconnus et le modèle risque de ne pas avoir de bonnes performances.

Dans le domaine de la "computer vision", on dit souvent que:
* les couches du début permettent de détecter des caractéristiques telles que les bordures et les couleurs
* les couches au milieu détectent la forme des objets
* les couches les plus profondes détectent les détails sur les objets

Cependant, ajouter de nombreuses couches n'est pas pour autant une bonne solution. Cela augmente de façon exponentielle le nombre d'hyperparamètres et peut rendre l'entraînement instable si on n'utilise pas des techniques telles que la "batch normalization" ou le "dropout".

In [7]:
class NeuralNetwork3Hidden(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir runs

In [ ]:
model_3_hidden = NeuralNetwork3Hidden()
SGD_optimizer = optim.SGD(model_3_hidden.parameters(), lr=1e-3)

pipeline = NewModelPipeline(
    model=model_3_hidden,
    dataset_fn=datasets.FashionMNIST,
    optimizer=SGD_optimizer,
    loss_fn=loss_x_entropy,
    batch_size=64,
    log_name="3_hidden_layers"
)

pipeline.fit(epochs=50)

Facile n'est-ce pas?
Dans cet exemple, ajouter un 3e hidden layer de 512 neurones n'a pas amélioré les performances. Au contraire ces dernières étaient inférieures au début de l'entraînement. 
Cela pourrait signifier que toutes les caractéristiques importantes étaient déjà capturées par le modèle précédent.
Ainsi, on n'aurait fait qu'augmenter la complexité du modèle et rendu moins stable.

In [8]:
class NeuralNetwork1024(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir runs

In [ ]:
model_1024_neurons = NeuralNetwork1024()
SGD_optimizer = optim.SGD(model_1024_neurons.parameters(), lr=1e-3)

pipeline = NewModelPipeline(
    model=model_1024_neurons,
    dataset_fn=datasets.FashionMNIST,
    optimizer=SGD_optimizer,
    loss_fn=loss_x_entropy,
    batch_size=64,
    log_name="1024_neurons"
)

pipeline.fit(epochs=50)

## Première analyse de résultats

Dans cet exemple nous avons doublé le nombre de neurones dans les hidden layers. 
Dans avions pu constaté que pour le modèle original avec la SGD, l'entraînement est lent et atteint une précision raisonnable.
Comme nous pouvions nous y attendre, augmenter la taille des couches dans ce cas n'a quasiment rien changé.
Ainsi il vaut mieux garder un nombre de paramètres plus faibles tel que 512.
![similar curve with 1024 neurons](../img/with_1024_neurons.png)

In [9]:
class NeuralNetwork128(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        x = self.flatten(x)
        logits = self.linear_relu_stack(x)
        return logits

In [ ]:
%reload_ext tensorboard
%tensorboard --logdir runs

In [ ]:
model_128_neurons = NeuralNetwork128()
SGD_optimizer = optim.SGD(model_128_neurons.parameters(), lr=1e-3)
pipeline = NewModelPipeline(
    model=model_128_neurons,
    dataset_fn=datasets.FashionMNIST,
    optimizer=SGD_optimizer,
    loss_fn=loss_x_entropy,
    batch_size=64,
    log_name="128_neurons"
)

pipeline.fit(epochs=50)

## Résultats pour 128 neurones seulement

Par rapport à ce qu'on a dit précédemment, nous pouvons également nous attendre à une baisse des performances en prenant un nombre de neurones trop faible.

Dans ce cas, avec 128 neurones, la courbe est tout de même similaire à l'originelle même si un peu moins performante.

![similar curve with 128 neurons](../img/with_128_neurons.png)

On peut donc conclure sur le fait que jouer avec ces paramètres peut avoir une influence sur le performances même si limitée tant que nous n'utilisons pas de valeurs trop abusives.